In [0]:
%sql
CREATE OR REPLACE TABLE `ct-esteira-dados-aviacao`.business.obt_voos_status
USING DELTA
AS
SELECT
    v.num_voo,
    v.empresa_aerea,
    a_origem.sigla_iata AS iata_origem,
    a_destino.sigla_iata AS iata_destino,
    (unix_timestamp(v.partida_real) - unix_timestamp(v.partida_prevista)) / 60 AS antc_dept,
    (unix_timestamp(v.chegada_real) - unix_timestamp(v.chegada_prevista)) / 60 AS antc_arr,
    
    CASE 
        WHEN unix_timestamp(v.partida_real) < unix_timestamp(v.partida_prevista) THEN 'ANTECIPADO'
        WHEN unix_timestamp(v.partida_real) > unix_timestamp(v.partida_prevista) THEN 'ATRASADO'
        ELSE 'NO HORÁRIO'
    END AS status_partida_calc,
    
    CASE 
        WHEN unix_timestamp(v.chegada_real) < unix_timestamp(v.chegada_prevista) THEN 'ANTECIPADO'
        WHEN unix_timestamp(v.chegada_real) > unix_timestamp(v.chegada_prevista) THEN 'ATRASADO'
        ELSE 'NO HORÁRIO'
    END AS status_chegada_calc,
    
    (unix_timestamp(v.chegada_real) - unix_timestamp(v.partida_real)) / 60 AS tempo_de_voo

FROM `ct-esteira-dados-aviacao`.trusted.dim_voos_trusted_dep_arr v
LEFT JOIN `ct-esteira-dados-aviacao`.trusted.dim_aeros_siglas_trusted a_origem
    ON v.sigla_aero_origem = a_origem.sigla_icao
LEFT JOIN `ct-esteira-dados-aviacao`.trusted.dim_aeros_siglas_trusted a_destino
    ON v.sigla_aero_destino = a_destino.sigla_icao
WHERE
    (unix_timestamp(v.partida_real) - unix_timestamp(v.partida_prevista)) IS NOT NULL
    AND (unix_timestamp(v.chegada_real) - unix_timestamp(v.chegada_prevista)) IS NOT NULL
    AND (
        v.sigla_cia_aerea IN ('LAN', 'GLO')  -- LATAM e GOL pelo código
        OR v.empresa_aerea LIKE 'AZUL%'       -- AZUL pelo nome
    )


In [0]:
%sql
OPTIMIZE `ct-esteira-dados-aviacao`.business.obt_voos_status
ZORDER BY (empresa_aerea, iata_origem, iata_destino);
